In [1]:
import yfinance as yf
import pandas as pd
import numpy as np
from pathlib import Path
import os
import requests
import pandas as pd
import time
from pathlib import Path

In [2]:
RAW_FILE = Path("../data/raw/djia30_tiingo_adjclose_2010_2026.csv")

prices = pd.read_csv(
    RAW_FILE,
    index_col=0,
    parse_dates=True
)

print("Shape:", prices.shape)
print("First date:", prices.index.min())
print("Last date:", prices.index.max())

prices.head()

Shape: (4169, 30)
First date: 2010-01-04 00:00:00+00:00
Last date: 2026-07-31 00:00:00+00:00


,AAPL,AMGN,AMZN,AXP,BA,CAT,CRM,CSCO,CVX,DIS,...,MRK,MSFT,NKE,NVDA,PG,SHW,TRV,UNH,V,WMT
date,,,,,,,,,,,,,,,,,,,,,
2010-01-04 00:00:00+00:00,6.400674,38.763960,6.6950,32.314584,43.712259,39.062197,18.381700,15.836508,40.897868,27.083542,...,20.330952,23.027318,13.096524,0.423362,37.557881,17.167371,34.064252,24.286690,19.495470,12.909962
2010-01-05 00:00:00+00:00,6.411740,38.431296,6.7345,32.243511,45.143917,39.529209,18.303083,15.765953,41.187557,27.015981,...,20.413352,23.034758,13.148629,0.429545,37.570171,16.760944,33.257270,24.248176,19.272071,12.781410
2010-01-06 00:00:00+00:00,6.309753,38.142490,6.6125,32.764714,46.513329,39.649297,18.271144,15.663327,41.192730,26.872414,...,20.688020,22.893395,13.068467,0.432292,37.391968,16.541028,32.785390,24.486961,19.013282,12.752843
2010-01-07 00:00:00+00:00,6.298089,37.793237,6.5000,33.293814,48.396271,39.809416,18.190070,15.733882,41.037540,26.880859,...,20.720981,22.656798,13.196727,0.423820,37.189185,16.688567,33.257270,25.426693,19.190232,12.759984
2010-01-08 00:00:00+00:00,6.339961,38.129058,6.6760,33.270021,47.929426,40.256413,18.217095,15.817266,41.109962,26.923085,...,20.709994,22.811553,13.170674,0.424736,37.140025,16.830538,33.209398,25.187909,19.243317,12.695708


In [3]:
missing_count = prices.isna().sum()

missing_pct = prices.isna().mean() * 100

first_valid_date = prices.apply(
    lambda column: column.first_valid_index()
)

last_valid_date = prices.apply(
    lambda column: column.last_valid_index()
)

In [4]:
print(missing_count)
print(missing_pct)

AAPL     0
AMGN     0
AMZN     0
AXP      0
BA       0
CAT      0
CRM      0
CSCO     0
CVX      0
DIS      0
GOOGL    0
GS       0
HD       0
HON      0
IBM      0
JNJ      0
JPM      0
KO       0
MCD      0
MMM      0
MRK      0
MSFT     0
NKE      0
NVDA     0
PG       0
SHW      0
TRV      0
UNH      0
V        0
WMT      0
dtype: int64
AAPL     0.0
AMGN     0.0
AMZN     0.0
AXP      0.0
BA       0.0
CAT      0.0
CRM      0.0
CSCO     0.0
CVX      0.0
DIS      0.0
GOOGL    0.0
GS       0.0
HD       0.0
HON      0.0
IBM      0.0
JNJ      0.0
JPM      0.0
KO       0.0
MCD      0.0
MMM      0.0
MRK      0.0
MSFT     0.0
NKE      0.0
NVDA     0.0
PG       0.0
SHW      0.0
TRV      0.0
UNH      0.0
V        0.0
WMT      0.0
dtype: float64


In [5]:
returns = prices / prices.shift(1) - 1 
returns = returns.iloc[1:].copy()
print("Returns shape:", returns.shape)
print("Total NaNs:", returns.isna().sum().sum())
print("Number of infinite values:", np.isinf(returns.to_numpy()).sum())

display(returns.head())
display(returns.describe().T)


Returns shape: (4168, 30)
Total NaNs: 0
Number of infinite values: 0


,AAPL,AMGN,AMZN,AXP,BA,CAT,CRM,CSCO,CVX,DIS,...,MRK,MSFT,NKE,NVDA,PG,SHW,TRV,UNH,V,WMT
date,,,,,,,,,,,,,,,,,,,,,
2010-01-05 00:00:00+00:00,0.001729,-0.008582,0.005900,-0.002199,0.032752,0.011956,-0.004277,-0.004455,0.007083,-0.002495,...,0.004053,0.000323,0.003979,0.014602,0.000327,-0.023674,-0.023690,-0.001586,-0.011459,-0.009958
2010-01-06 00:00:00+00:00,-0.015906,-0.007515,-0.018116,0.016165,0.030334,0.003038,-0.001745,-0.006509,0.000126,-0.005314,...,0.013455,-0.006137,-0.006097,0.006397,-0.004743,-0.013121,-0.014189,0.009848,-0.013428,-0.002235
2010-01-07 00:00:00+00:00,-0.001849,-0.009157,-0.017013,0.016148,0.040482,0.004038,-0.004437,0.004505,-0.003767,0.000314,...,0.001593,-0.010335,0.009814,-0.019597,-0.005423,0.008920,0.014393,0.038377,0.009307,0.000560
2010-01-08 00:00:00+00:00,0.006648,0.008886,0.027077,-0.000715,-0.009646,0.011228,0.001486,0.005300,0.001765,0.001571,...,-0.000530,0.006830,-0.001974,0.002161,-0.001322,0.008507,-0.001439,-0.009391,0.002766,-0.005037
2010-01-11 00:00:00+00:00,-0.008822,0.004404,-0.024056,-0.011442,-0.011851,0.062811,-0.006743,-0.002839,0.017743,-0.016311,...,0.003979,-0.012720,-0.012325,-0.014016,-0.003971,-0.013728,-0.000412,0.006728,-0.002874,0.016501


,count,mean,std,min,25%,50%,75%,max
AAPL,4168.0,0.001088,0.017743,-0.128647,-0.007395,0.000998,0.010258,0.153288
AMGN,4168.0,0.000670,0.015444,-0.095846,-0.007270,0.000369,0.008582,0.118180
AMZN,4168.0,0.001104,0.020776,-0.140494,-0.009192,0.000922,0.012024,0.157457
AXP,4168.0,0.000729,0.018338,-0.148187,-0.007112,0.000748,0.009380,0.218823
BA,4168.0,0.000639,0.022577,-0.238484,-0.009253,0.000636,0.010527,0.243186
CAT,4168.0,0.000906,0.018803,-0.142822,-0.008533,0.000702,0.010678,0.116346
CRM,4168.0,0.000815,0.022946,-0.197371,-0.010042,0.000736,0.011850,0.260449
CSCO,4168.0,0.000616,0.016570,-0.162107,-0.006488,0.000614,0.008236,0.159505
CVX,4168.0,0.000518,0.016762,-0.221248,-0.007286,0.000777,0.008493,0.227407
DIS,4168.0,0.000442,0.016618,-0.131632,-0.006983,0.000308,0.008260,0.144123


In [6]:
#duplicates test
duplicate_dates = returns.index.duplicated().sum()

print("Duplicate dates:", duplicate_dates)

Duplicate dates: 0


In [7]:
#Date order test
print(
    "Dates sorted:",
    returns.index.is_monotonic_increasing
)

Dates sorted: True


In [8]:
extreme_table = (
    returns
    .stack()
    .rename("return")
    .reset_index()
)

extreme_table.columns = [
    "date",
    "ticker",
    "return"
]

extreme_table["abs_return"] = (
    extreme_table["return"].abs()
)

extreme_table = extreme_table.sort_values(
    "abs_return",
    ascending=False
)

display(extreme_table.head(20))

,date,ticker,return,abs_return
51833,2016-11-11 00:00:00+00:00,NVDA,0.298067,0.298067
80376,2020-08-26 00:00:00+00:00,CRM,0.260449,0.260449
124634,2026-07-14 00:00:00+00:00,IBM,-0.252076,0.252076
101123,2023-05-25 00:00:00+00:00,NVDA,0.243696,0.243696
77164,2020-03-25 00:00:00+00:00,BA,0.243186,0.243186
76954,2020-03-16 00:00:00+00:00,BA,-0.238484,0.238484
109909,2024-07-26 00:00:00+00:00,MMM,0.229906,0.229906
77138,2020-03-24 00:00:00+00:00,CVX,0.227407,0.227407
115377,2025-04-17 00:00:00+00:00,UNH,-0.223797,0.223797
77018,2020-03-18 00:00:00+00:00,CVX,-0.221248,0.221248


In [9]:
large_moves = (
    extreme_table[
        extreme_table["abs_return"] >= 0.20
    ]
)

print(
    "Observations with |daily return| >= 20%:",
    len(large_moves)
)

display(large_moves)

Observations with |daily return| >= 20%: 14


,date,ticker,return,abs_return
51833,2016-11-11 00:00:00+00:00,NVDA,0.298067,0.298067
80376,2020-08-26 00:00:00+00:00,CRM,0.260449,0.260449
124634,2026-07-14 00:00:00+00:00,IBM,-0.252076,0.252076
101123,2023-05-25 00:00:00+00:00,NVDA,0.243696,0.243696
77164,2020-03-25 00:00:00+00:00,BA,0.243186,0.243186
76954,2020-03-16 00:00:00+00:00,BA,-0.238484,0.238484
109909,2024-07-26 00:00:00+00:00,MMM,0.229906,0.229906
77138,2020-03-24 00:00:00+00:00,CVX,0.227407,0.227407
115377,2025-04-17 00:00:00+00:00,UNH,-0.223797,0.223797
77018,2020-03-18 00:00:00+00:00,CVX,-0.221248,0.221248


In [10]:
return_range = pd.DataFrame({
    "min_return": returns.min(),
    "max_return": returns.max()
})

return_range["max_abs_return"] = (
    returns.abs().max()
)

return_range = return_range.sort_values(
    "max_abs_return",
    ascending=False
)

display(return_range)

,min_return,max_return,max_abs_return
NVDA,-0.187559,0.298067,0.298067
CRM,-0.197371,0.260449,0.260449
IBM,-0.252076,0.129642,0.252076
BA,-0.238484,0.243186,0.243186
MMM,-0.129450,0.229906,0.229906
CVX,-0.221248,0.227407,0.227407
UNH,-0.223797,0.127989,0.223797
AXP,-0.148187,0.218823,0.218823
TRV,-0.208004,0.132902,0.208004
NKE,-0.199809,0.155314,0.199809


In [11]:
zero_mask = np.isclose(
    returns.to_numpy(),
    0.0,
    atol=1e-12
)

zero_counts = pd.Series(
    zero_mask.sum(axis=0),
    index=returns.columns,
    name="zero_return_days"
)

zero_counts.sort_values(ascending=False)

CSCO     48
KO       44
MRK      33
PG       27
JNJ      26
MSFT     25
TRV      24
DIS      22
MCD      21
JPM      20
NVDA     20
MMM      19
WMT      19
NKE      18
UNH      14
AXP      13
HD       13
CVX      11
HON      11
V        11
IBM      10
CRM      10
SHW      10
AMZN      9
BA        9
AAPL      7
AMGN      6
CAT       6
GS        4
GOOGL     3
Name: zero_return_days, dtype: int64

In [12]:
def longest_zero_run(series):
    zero = np.isclose(series.to_numpy(), 0.0, atol=1e-12)

    longest = 0
    current = 0

    for is_zero in zero:
        if is_zero:
            current += 1
            longest = max(longest, current)
        else:
            current = 0

    return longest


longest_zero_runs = returns.apply(longest_zero_run)

zero_summary = pd.DataFrame({
    "zero_return_days": zero_counts,
    "longest_zero_run": longest_zero_runs
})

zero_summary = zero_summary.sort_values(
    "longest_zero_run",
    ascending=False
)

display(zero_summary)

,zero_return_days,longest_zero_run
BA,9,2
AMZN,9,2
HD,13,2
CSCO,48,2
KO,44,2
JNJ,26,2
MMM,19,2
CAT,6,1
CVX,11,1
CRM,10,1
